# Diferenciación automática

**Capítulo 2 · Universidad de las Hespérides**

Adaptación al español de *Dive into Deep Learning*, Aston Zhang, Zachary C. Lipton, Mu Li y Alexander J. Smola.
Fuente: `locked/chapter_preliminaries/autograd.ipynb` · [Lección original](https://d2l.ai/chapter_preliminaries/autograd.html).
Texto adaptado bajo [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). [Procedencia y cambios](../PROCEDENCIA.md).
Se conserva la secuencia de las celdas y de los ejercicios; las notas de Hespérides se identifican expresamente.

**Entorno:** ejecuta `uv sync` en la raíz y selecciona su Python como kernel. Las descargas se realizan una vez y quedan en `data/`.
Por defecto, el soporte limita los entrenamientos de `Trainer` a tres épocas y 1024/256 ejemplos para CPU.
Para repetir el régimen completo, inicia Jupyter con `HESPERIDES_COMPLETO=1`. Los ejemplos visuales pequeños conservan su propia configuración explícita.
Los datos de texto en inglés o francés son entradas de los experimentos originales y mantienen su idioma.


In [ ]:
from pathlib import Path
import sys
RAIZ = Path.cwd() if (Path.cwd() / "laboratorio").exists() else Path.cwd().parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
from laboratorio import d2l, configurar, epocas
configurar()


# Diferenciación automática
<a id="sec_autograd"></a>

Como vimos en [Referencia sec_calculus](https://d2l.ai/chapter_preliminaries/calculus.html#sec-calculus), calcular derivadas es un paso esencial de los métodos de optimización con los que entrenaremos redes. Hacerlo a mano puede resultar tedioso y propenso a errores, especialmente cuando el modelo combina muchas operaciones.

Las bibliotecas de Deep Learning proporcionan *diferenciación automática*, habitualmente denominada *autograd*. Durante el cálculo registran un grafo que describe cómo depende cada valor de los anteriores. Para una pérdida escalar, la diferenciación en modo inverso recorre ese grafo hacia atrás y aplica la regla de la cadena. En redes neuronales, este procedimiento recibe el nombre de *retropropagación*.

Aunque estas herramientas han ganado protagonismo recientemente, sus ideas tienen una historia larga. Las referencias tempranas incluyen [Wengert.1964](https://d2l.ai/chapter_references/zreferences.html), la tesis de [Speelpenning.1980](https://d2l.ai/chapter_references/zreferences.html) y los trabajos de [Griewank.1989](https://d2l.ai/chapter_references/zreferences.html). El modo inverso no es la única opción: también existe diferenciación en modo directo, disponible, por ejemplo, en herramientas del ecosistema Julia [Revels.Lubin.Papamarkou.2016](https://d2l.ai/chapter_references/zreferences.html). Antes de comparar sus costes, aprenderemos a utilizar autograd en PyTorch.


In [ ]:
import torch

## Una función sencilla
Supongamos que estamos interesados en **diferenciar la función $y = 2\mathbf{x}^{\top}\mathbf{x}$ con respecto al vector de columna $\mathbf{x}$.** Para empezar, asignamos `x` un valor inicial.


In [ ]:
x = torch.arange(4.0)
x

** Antes de calcular el gradiente de $y$ con respecto a $\mathbf{x}$, necesitamos un lugar para almacenarlo.** En general, evitamos asignar nueva memoria cada vez que tomamos una derivada porque el aprendizaje profundo requiere sucesivamente computar derivados con respecto a los mismos parámetros muchas veces, y podríamos correr el riesgo de quedarse sin memoria. Tenga en cuenta que el gradiente de una función de valor escalar con respecto a un vector $\mathbf{x}$ es vector-valorado con la misma forma que $\mathbf{x}$.


In [ ]:
# También se puede crear x = torch.arange(4.0, requires_grad=True)
x.requires_grad_(True)
x.grad  # El gradiente es Ninguno por defecto

**Ahora calculamos nuestra función de `x` y asignamos el resultado a `y`.**


In [ ]:
y = 2 * torch.dot(x, x)
y

**Ahora podemos tomar el gradiente de `y` con respecto a `x`** llamando a su método `backward`. A continuación, podemos acceder al gradiente a través del atributo `grad` de `x`.


In [ ]:
y.backward()
x.grad

**Ya sabemos que el gradiente de la función $y = 2\mathbf{x}^{\top}\mathbf{x}$ con respecto a $\mathbf{x}$ debe ser $4\mathbf{x}$.** Ahora podemos verificar que el cálculo automático del gradiente y el resultado esperado son idénticos.


In [ ]:
x.grad == 4 * x

**Ahora vamos a calcular otra función de `x` y tomar su gradiente.** Tenga en cuenta que PyTorch no restablece automáticamente el buffer de gradiente cuando registramos un nuevo gradiente. En su lugar, el nuevo gradiente se añade al gradiente ya almacenado. Este comportamiento es útil cuando queremos optimizar la suma de múltiples funciones objetivas. Para restablecer el gradiente buffer, podemos llamar a `x.grad.zero_()` de la siguiente manera:


In [ ]:
x.grad.zero_()  # Reiniciar el gradiente
y = x.sum()
y.backward()
x.grad

### Nota docente de Hespérides

Autograd aplica la regla de la cadena sobre las operaciones ejecutadas. Para una pérdida escalar, una pasada inversa calcula sus derivadas respecto de muchos parámetros. Los gradientes se acumulan en `.grad`: ponerlos a cero separa pasos de optimización; conservar el grafo es otra decisión. Comprueba tres cosas por separado: `requires_grad`, existencia de una conexión con la pérdida y reinicio de los gradientes. Un tensor desconectado puede tener una forma impecable y no aprender.

Vínculo con los apuntes: sesión 2, «Diferenciación automática».


## Retropropagación con variables no escalares
Cuando `y` es un vector, la representación más natural de la derivada de `y` con respecto a un vector `x` es una matriz llamada *Jacobiana* que contiene las derivadas parciales de cada componente de `y` con respecto a cada componente de `x`. Del mismo modo, para `y` de mayor orden y `x`, el resultado de la diferenciación podría ser un tensor de mayor orden aún.

Mientras que los jacobinos aparecen en algunas técnicas avanzadas de aprendizaje automático, más comúnmente queremos resumir los gradientes de cada componente de `y` con respecto al vector `x` completo, dando un vector de la misma forma que `x`. Por ejemplo, a menudo tenemos un vector que representa el valor de nuestra función de pérdida calculado por separado para cada ejemplo entre un *batch* de ejemplos de entrenamiento. Aquí, sólo queremos **sumar los gradientes calculados individualmente para cada ejemplo**.


Debido a que los bibliotecas de aprendizaje profundo varían en cómo interpretan los gradientes de los tensores no escalares, PyTorch toma algunos pasos para evitar confusión. Invocar `backward` en un no escalar provoca un error a menos que le digamos a PyTorch cómo reducir el objeto a un escalar. Más formalmente, necesitamos proporcionar algún vector $\mathbf{v}$ tal que `backward` calcule $\mathbf{v}^\top \partial_{\mathbf{x}} \mathbf{y}$ en lugar de $\partial_{\mathbf{x}} \mathbf{y}$. Esta siguiente parte puede ser confusa, pero por razones que se aclararán más adelante, este argumento (representando $\mathbf{v}$) se llama `gradient`. Para una descripción más detallada, véase la [Medium post](https://zhang-yang.medium.com/the-gradient-argument-in-pytorchs-backward-function-explained-by-examples-68f266950c29) de Yang Zhang.


In [ ]:
x.grad.zero_()
y = x * x
y.backward(gradient=torch.ones(len(y)))  # Más rápido: y.sum().backward()
x.grad

## Desacoplar del grafo computacional
A veces, queremos **mover algunos cálculos fuera del grafo computacional registrado.** Por ejemplo, digamos que usamos la entrada para crear algunos términos intermedios auxiliares para los que no queremos calcular un gradiente. En este caso, necesitamos *destacar* el grafo computacional respectivo del resultado final. El siguiente ejemplo de juguete lo aclara: supongamos que tenemos `z = x * y` y `y = x * x` pero queremos centrarnos en la influencia *directa* de `x` en `z` en lugar de la influencia transmitida a través de `y`. En este caso, podemos crear una nueva variable `u` que tome el mismo valor que `y` pero cuya *provenencia* (cómo se creó) ha sido borrada. Así, `u` no tiene antepasados en el gráfico y los gradientes no fluyen a través de `u` a `x`. Por ejemplo, tomando el gradiente de `z = x * u` producirá el resultado `u`, (no `3 * x * x` como usted podría haber esperado desde `z = x * x * x`).


In [ ]:
x.grad.zero_()
y = x * x
u = y.detach()
z = u * x

z.sum().backward()
x.grad == u

Tenga en cuenta que mientras este procedimiento separa a los antepasados de `y` del gráfico que conduce a `z`, el grafo computacional que conduce a `y` persiste y por lo tanto podemos calcular el gradiente de `y` con respecto a `x`.


In [ ]:
x.grad.zero_()
y.sum().backward()
x.grad == 2 * x

## Gradientes y flujo de control de Python
Hasta ahora hemos revisado casos donde la ruta de entrada a salida estaba bien definida a través de una función como `z = x * x * x`. La programación nos ofrece mucha más libertad en cómo calculamos los resultados. Por ejemplo, podemos hacer que dependan de variables auxiliares o opciones de condición en resultados intermedios. Un beneficio de usar diferenciación automática es que **incluso si** construyendo la gráfica computacional de **una función requerida pasando a través de un laberinto de flujo de control de Python** (por ejemplo, condicionales, bucles y llamadas arbitrarias a funciones), **todavía podemos calcular el gradiente de la variable resultante.** Para ilustrar esto, considere el siguiente fragmento de código donde el número de iteraciones del bucle `while` y la evaluación de la instrucción `if` dependen del valor de la entrada `a`.


In [ ]:
def f(a):
    b = a * 2
    while b.norm() < 1000:
        b = b * 2
    if b.sum() > 0:
        c = b
    else:
        c = 100 * b
    return c

A continuación, llamamos a esta función, pasando en un valor aleatorio, como entrada. Puesto que la entrada es una variable aleatoria, no sabemos qué forma tomará el grafo computacional. Sin embargo, cada vez que ejecutamos `f(a)` en una entrada específica, realizamos un grafo computacional específico y posteriormente podemos ejecutar `backward`.


In [ ]:
a = torch.randn(size=(), requires_grad=True)
d = f(a)
d.backward()

A pesar de que nuestra función `f` es, para fines de demostración, un poco artificial, su dependencia de la entrada es bastante simple: es una función *lineal* de `a` con escala definida a trozos. Como tal, `f(a) / a` es un vector de entradas constantes y, además, `f(a) / a` necesita igualar el gradiente de `f(a)` con respecto a `a`.


In [ ]:
a.grad == d / a

El flujo de control dinámico es muy común en el aprendizaje profundo. Por ejemplo, cuando se procesa el texto, el grafo computacional depende de la longitud de la entrada. En estos casos, la diferenciación automática se convierte en vital para el modelado estadístico, ya que es imposible calcular el gradiente *a priori*.

## Discusión
El desarrollo de bibliotecas para calcular derivadas de forma automática y eficiente ha sido un impulsor masivo de productividad para los profesionales del aprendizaje profundo, liberándolos para que puedan centrarse en menos menial. Además, autograd nos permite diseñar modelos masivos para los cuales los cálculos de gradiente de lápiz y papel serían prohibitivos. Curiosamente, mientras usamos autograd para *optimizar* modelos (en un sentido estadístico) la *optimización* de las propias bibliotecas de autograd (en un sentido computacional) es un tema rico de interés vital para los diseñadores de marcos. Aquí, herramientas de compiladores y manipulación de gráficos se aprovechan para calcular resultados de la manera más conveniente y eficiente de la memoria.

Por ahora, trate de recordar estos conceptos básicos: (i) adjuntar gradientes a esas variables con respecto a las cuales deseamos derivados; (ii) registrar el cálculo del valor objetivo; (iii) ejecutar la función de retropropagación; y (iv) acceder al gradiente resultante.

## Ejercicios
1. ¿Por qué es la segunda derivada mucho más caro de calcular que la primera derivada?
1. Después de ejecutar la retropropagación, vuelve a ejecutarla inmediatamente y observa qué sucede. Explica el resultado.
1. En el ejemplo de flujo de control donde calculamos la derivada de `d` con respecto a `a`, ¿qué pasaría si cambiamos la variable `a` a un vector aleatorio o una matriz? En este punto, el resultado del cálculo `f(a)` ya no es escalar. ¿Qué sucede con el resultado? ¿Cómo analizamos esto?
1. Sea $f(x) = \sin(x)$. Traza el gráfico de $f$ y de su derivada $f'$. No utilices el hecho de que $f'(x) = \cos(x)$, sino utiliza la diferenciación automática para obtener el resultado.
1. Sea $f(x) = ((\log x^2) \cdot \sin x) + x^{-1}$. Escriba un grafo de dependencias que siga el cálculo de $x$ a $f(x)$.
1. Utilice la regla de cadena para calcular la derivada $\frac{df}{dx}$ de la función antes mencionada, colocando cada término en el gráfico de dependencia que construyó previamente.
1. Dado el gráfico y los derivadas intermedias, usted tiene una serie de opciones al calcular el gradiente. Evalúe el resultado una vez que comienza desde $x$ a $f$ y una vez desde $f$ trazando de nuevo a $x$. La ruta de $x$ a $f$ se conoce comúnmente como *diferenciación hacia adelante*, mientras que la ruta de $f$ a $x$ se conoce como diferenciación hacia atrás.
1. ¿Cuándo podría utilizar hacia adelante, y cuando hacia atrás, diferenciación? Consejo: considere la cantidad de datos intermedios necesarios, la capacidad de paralelizar pasos, y el tamaño de matrices y vectores involucrados.


[Debate del original](https://discuss.d2l.ai/t/35)
